In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkAdvancedOps")\
        .master("spark://spark-master:7077").getOrCreate() 
df = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferSchema", "true")\
          .load("/home/jovyan/data/online-retail-dataset.csv")\
          .coalesce(5)   # The coalesce() is used to decrease the number of partitions in an efficient way.
df.cache() # Yields and caches the current DataFrame.
df.printSchema()
df.show(10, False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/2

We can use the function **count** to count the number of items (rows) in a group defined by one or more columns. The function **countDistinct** is for counting the number of distinct items in a group.

In [2]:
from pyspark.sql.functions import count
df.select(count("StockCode")).show()

+----------------+
|count(StockCode)|
+----------------+
|          541909|
+----------------+



In [3]:
from pyspark.sql.functions import countDistinct
df.select(countDistinct("StockCode")).show()

+-------------------------+
|count(DISTINCT StockCode)|
+-------------------------+
|                     4070|
+-------------------------+



We can also use an approximation of the distinct count. This is efficient compared with counting all. 

In [4]:
from pyspark.sql.functions import approx_count_distinct
df.select(approx_count_distinct("StockCode", 0.1)).show()    # 0.1 maximum relative standard deviation allowed

+--------------------------------+
|approx_count_distinct(StockCode)|
+--------------------------------+
|                            3364|
+--------------------------------+



We can also get first, last, minimum, and maximum values from a group (one or more columns) in a DataFrame. 

In [5]:
from pyspark.sql.functions import first, last, min, max

df.select(first("StockCode"), last("StockCode")).show()
df.select(min("Quantity"), max("Quantity")).show()

+----------------+---------------+
|first(StockCode)|last(StockCode)|
+----------------+---------------+
|          85123A|          22138|
+----------------+---------------+

+-------------+-------------+
|min(Quantity)|max(Quantity)|
+-------------+-------------+
|       -80995|        80995|
+-------------+-------------+



We can also compute the sum for each numeric columns for each group.

In [6]:
from pyspark.sql.functions import sum, sum_distinct

df.select(sum("Quantity")).show() 
df.select(sum_distinct("Quantity")).show() 

+-------------+
|sum(Quantity)|
+-------------+
|      5176450|
+-------------+

+----------------------+
|sum(DISTINCT Quantity)|
+----------------------+
|                 29310|
+----------------------+



We can also compute the average value for each numeric column within each group. mean() is an alias for avg(). Note that each select clause returns a new data frame. 

In [7]:
from pyspark.sql.functions import sum, count, avg, expr

df1 = df.select(
    count("Quantity").alias("total_transactions"),
    sum("Quantity").alias("total_purchases"),
    avg("Quantity").alias("avg_purchases"),
    expr("mean(Quantity)").alias("mean_purchases"))

df1.show(3)

df1.selectExpr(
    "total_purchases/total_transactions",
    "avg_purchases",
    "mean_purchases").show()

+------------------+---------------+----------------+----------------+
|total_transactions|total_purchases|   avg_purchases|  mean_purchases|
+------------------+---------------+----------------+----------------+
|            541909|        5176450|9.55224954743324|9.55224954743324|
+------------------+---------------+----------------+----------------+

+--------------------------------------+----------------+----------------+
|(total_purchases / total_transactions)|   avg_purchases|  mean_purchases|
+--------------------------------------+----------------+----------------+
|                      9.55224954743324|9.55224954743324|9.55224954743324|
+--------------------------------------+----------------+----------------+



Spark has both the formula for the sample standard deviation and the formula for the population standard deviation.

In [8]:
from pyspark.sql.functions import var_pop, stddev_pop
from pyspark.sql.functions import var_samp, stddev_samp

df.select(var_pop("Quantity"), var_samp("Quantity"),
  stddev_pop("Quantity"), stddev_samp("Quantity")).show()

+------------------+------------------+--------------------+---------------------+
| var_pop(Quantity)|var_samp(Quantity)|stddev_pop(Quantity)|stddev_samp(Quantity)|
+------------------+------------------+--------------------+---------------------+
|47559.303646609354| 47559.39140929905|  218.08095663447864|   218.08115785023486|
+------------------+------------------+--------------------+---------------------+



Spark provides a formula for computing the skewness and kurtosis of a group's values. Skewness measures the asymmetry of the values in your data around the mean. Kurtosis is a measure of the tails of the data distribution.

In [9]:
from pyspark.sql.functions import skewness, kurtosis
df.select(skewness("Quantity"), kurtosis("Quantity")).show()

+--------------------+------------------+
|  skewness(Quantity)|kurtosis(Quantity)|
+--------------------+------------------+
|-0.26407557610527843|119768.05495536518|
+--------------------+------------------+



We can calculate the correlation of two columns of a DataFrame as a double value. Correlation measures the Pearson correlation coefficient, which ranges from -1 to +1.
We can also compute the sample and population covariance of two columns.

In [16]:
from pyspark.sql.functions import corr, covar_pop, covar_samp, col
from pyspark.sql import functions as F

# Clean out InvoiceNos that are  not an integer
# [^\d.] matches anything that is NOT a digit or a decimal point
df_cleaned = df.withColumn(
    "InvoiceNo", 
    F.regexp_replace(F.col("InvoiceNo"), r"[^\d.]", "")
)


df_cleaned.select(var_pop("Quantity"), var_samp("Quantity"),
  stddev_pop("Quantity"), stddev_samp("Quantity")).show()


df_cleaned.show()
df_cleaned.printSchema()

df_cleaned.select(corr("InvoiceNo", "Quantity"), covar_samp("InvoiceNo", "Quantity"),
  covar_pop("InvoiceNo", "Quantity")).show()

+------------------+------------------+--------------------+---------------------+
| var_pop(Quantity)|var_samp(Quantity)|stddev_pop(Quantity)|stddev_samp(Quantity)|
+------------------+------------------+--------------------+---------------------+
|47559.303646609354| 47559.39140929905|  218.08095663447864|   218.08115785023486|
+------------------+------------------+--------------------+---------------------+

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|   

We do aggregations on complex types (collection).
collect_list- returns a list of objects with duplicates.
collect_set - returns a set of objects with duplicate elements eliminated.

In [17]:
from pyspark.sql.functions import collect_set, collect_list

df.agg(collect_set("Country"), collect_list("Country")).show(2)

+--------------------+---------------------+
|collect_set(Country)|collect_list(Country)|
+--------------------+---------------------+
|[Netherlands, Sau...| [United Kingdom, ...|
+--------------------+---------------------+



We can use the groupby clause (like SQL's group by) to group the data on one or more columns and perform calculations on the other columns that end up in that group. 

In [18]:
df.groupBy("InvoiceNo", "CustomerId").count().show()

+---------+----------+-----+
|InvoiceNo|CustomerId|count|
+---------+----------+-----+
|   563017|     13198|   32|
|   563214|     16370|   71|
|   563372|     15653|    4|
|   563714|     14565|   31|
|   564666|     12492|    9|
|  C565313|     14527|    1|
|   565318|     12921|   23|
|  C565962|     14410|   11|
|   566023|     12955|   18|
|   566079|     17593|   37|
|   566904|     15660|   19|
|   567476|     14859|   32|
|  C567643|     12409|    1|
|   568330|     16468|    2|
|   568509|     16422|    4|
|  C568911|     15050|    1|
|   569105|     16729|   49|
|   569211|     15774|   25|
|   569388|     14031|    1|
|   570179|     17509|   54|
+---------+----------+-----+
only showing top 20 rows


We normally do aggregations over groups. We can pass-in arbitrary expressions with agg function. 

In [19]:
from pyspark.sql.functions import count

df.groupBy("InvoiceNo").agg(
  count("Quantity").alias("quan"), # give the column name quan for count values
  expr("count(Quantity)")).show()

+---------+----+---------------+
|InvoiceNo|quan|count(Quantity)|
+---------+----+---------------+
|   563020|  24|             24|
|   565747|  10|             10|
|   566248|   8|              8|
|   566431|  18|             18|
|   567163|  14|             14|
|   567695|   1|              1|
|   567879|  29|             29|
|   568222|  11|             11|
|   568711|   4|              4|
|   569020|  48|             48|
|   569560|  16|             16|
|   569823|  69|             69|
|   570234|  36|             36|
|   570264|   1|              1|
|   570281|   3|              3|
|   570592|  73|             73|
|   571010|   1|              1|
|   571906|   1|              1|
|   572049|  20|             20|
|   572458|  26|             26|
+---------+----+---------------+
only showing top 20 rows


In [20]:
df.groupBy("InvoiceNo").agg(expr("avg(Quantity)"),expr("stddev_pop(Quantity)"))\
.show()

+---------+------------------+--------------------+
|InvoiceNo|     avg(Quantity)|stddev_pop(Quantity)|
+---------+------------------+--------------------+
|   563020|16.041666666666668|  10.212244148841895|
|   565747|              10.3|   3.163858403911275|
|   566248|               9.0|   8.703447592764606|
|   566431| 14.11111111111111|   6.911254017815104|
|   567163|              14.5|  11.280514172678478|
|   567695|             -42.0|                 0.0|
|   567879|10.206896551724139|  6.4774372877616235|
|   568222|10.636363636363637|   2.705977466570403|
|   568711|             15.25|   9.575359001102779|
|   569020|11.541666666666666|   8.467974800518848|
|   569560|            10.625|  3.7893765978060294|
|   569823|1.4782608695652173|  0.9869980199409517|
|   570234|3.5833333333333335|   2.542691050398726|
|   570264|             -22.0|                 0.0|
|   570281|              48.0|                 0.0|
|   570592| 7.438356164383562|  11.910610328345681|
|   571010| 

**Window Functions**

We can use window functions to perform unique aggregations by computing an aggregation over a specific “window” of data. A window function calculates a return value for every input row of a table based on a group of rows, called a frame. Each row can fall into one or more frames. A common use case is to look at a rolling average of a value, with each row representing one day. Each row would end up in seven different frames.

The first step is to define the window. We can use the Window class, which has three methods.
partitionBy(*cols) - divide into groups
orderBy(*cols) - ordering within the group
rowsBetween(start, end) - the frame specification (the rowsBetween statement) states which rows will be included in the frame. In the following example, all previous rows up to the current row are included.

In [21]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import desc

dfWithDate = df.withColumn("date", to_date(col("InvoiceDate"), "M/d/y H:mm")) # our dataframe with date column

windowSpec = Window\
  .partitionBy("CustomerId", "date")\
  .orderBy(desc("Quantity"))\
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

Now, we can apply window functions over the defined window. Spark supports three types of window functions: ranking, analytic, and aggregate. Let’s consider the following example.
An aggregate function - find the maximum purchase quantity over all time
A ranking function - determine which date had the maximum purchase quantity for every customer.

In [22]:
from pyspark.sql.functions import max, dense_rank, rank

maxPurchaseQuantity = max(col("Quantity")).over(windowSpec)
purchaseDenseRank = dense_rank().over(windowSpec)
purchaseRank = rank().over(windowSpec)


The above functions return columns (or expressions), which can be used in a DataFrame select statement

In [23]:
dfWithDate.where("CustomerId IS NOT NULL").orderBy("CustomerId")\
  .select(
    col("CustomerId"),
    col("date"),
    col("Quantity"),
    purchaseRank.alias("quantityRank"),
    purchaseDenseRank.alias("quantityDenseRank"),
    maxPurchaseQuantity.alias("maxPurchaseQuantity")).show(40)

+----------+----------+--------+------------+-----------------+-------------------+
|CustomerId|      date|Quantity|quantityRank|quantityDenseRank|maxPurchaseQuantity|
+----------+----------+--------+------------+-----------------+-------------------+
|     12346|2011-01-18|   74215|           1|                1|              74215|
|     12346|2011-01-18|  -74215|           2|                2|              74215|
|     12347|2010-12-07|      36|           1|                1|                 36|
|     12347|2010-12-07|      30|           2|                2|                 36|
|     12347|2010-12-07|      24|           3|                3|                 36|
|     12347|2010-12-07|      12|           4|                4|                 36|
|     12347|2010-12-07|      12|           4|                4|                 36|
|     12347|2010-12-07|      12|           4|                4|                 36|
|     12347|2010-12-07|      12|           4|                4|             

### Join Operations

We can perform join operations on DataFrames. In SQL, join table operations combine columns from one or more tables in a relational database. Here, Dataframes instead of tables. The following dataset is for learning join operations.

https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html 

In [25]:
person = spark.createDataFrame([
    (0, "Bill Chambers", 0, [100]),
    (1, "Matei Zaharia", 1, [500, 250, 100]),
    (2, "Michael Armbrust", 1, [250, 100])])\
    .toDF("id", "name", "graduate_program", "spark_status")

graduateProgram = spark.createDataFrame([
    (0, "Masters", "School of Information", "UC Berkeley"),
    (2, "Masters", "EECS", "UC Berkeley"),
    (1, "Ph.D.", "EECS", "UC Berkeley")])\
    .toDF("id", "degree", "department", "school")

sparkStatus = spark.createDataFrame([
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributor")])\
    .toDF("id", "status")

In [26]:
joinExpression = person["graduate_program"] == graduateProgram['id']
person.join(graduateProgram, joinExpression,"inner").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [27]:
person.join(graduateProgram, joinExpression,"inner").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [28]:
person.join(graduateProgram, joinExpression, "outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [29]:
graduateProgram.join(person, joinExpression, "left_outer").show()

+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
| id| degree|          department|     school|  id|            name|graduate_program|   spark_status|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
|  1|  Ph.D.|                EECS|UC Berkeley|   2|Michael Armbrust|               1|     [250, 100]|
|  1|  Ph.D.|                EECS|UC Berkeley|   1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Masters|                EECS|UC Berkeley|NULL|            NULL|            NULL|           NULL|
|  0|Masters|School of Informa...|UC Berkeley|   0|   Bill Chambers|               0|          [100]|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+



In [30]:
person.join(graduateProgram, joinExpression, "right_outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [31]:
graduateProgram.join(person, joinExpression, "left_anti").show()

+---+-------+----------+-----------+
| id| degree|department|     school|
+---+-------+----------+-----------+
|  2|Masters|      EECS|UC Berkeley|
+---+-------+----------+-----------+



In [32]:
graduateProgram.join(person, joinExpression, "left_semi").show()

+---+-------+--------------------+-----------+
| id| degree|          department|     school|
+---+-------+--------------------+-----------+
|  0|Masters|School of Informa...|UC Berkeley|
|  1|  Ph.D.|                EECS|UC Berkeley|
+---+-------+--------------------+-----------+



In [33]:
graduateProgram.join(person, joinExpression, "cross").show()

graduateProgram.crossJoin(person).show()

+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
| id| degree|          department|     school| id|            name|graduate_program|   spark_status|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
|  0|Masters|School of Informa...|UC Berkeley|  0|   Bill Chambers|               0|          [100]|
|  1|  Ph.D.|                EECS|UC Berkeley|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  1|  Ph.D.|                EECS|UC Berkeley|  2|Michael Armbrust|               1|     [250, 100]|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+

+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
| id| degree|          department|     school| id|            name|graduate_program|   spark_status|
+---+-------+--------------------+-----------+---+----------------+----------------+------

We can also perform joins based on complex types by using join expressions. Any expression is a valid join expression as long as it returns a Boolean.
In the following example, the join expression checks if “id” is in the “spark_status” column data (which is an array).

In [34]:
from pyspark.sql.functions import expr

person.withColumnRenamed("id", "personId")\
    .join(sparkStatus, expr("array_contains(spark_status, id)")).show()


+--------+----------------+----------------+---------------+---+--------------+
|personId|            name|graduate_program|   spark_status| id|        status|
+--------+----------------+----------------+---------------+---+--------------+
|       0|   Bill Chambers|               0|          [100]|100|   Contributor|
|       1|   Matei Zaharia|               1|[500, 250, 100]|500|Vice President|
|       1|   Matei Zaharia|               1|[500, 250, 100]|250|    PMC Member|
|       1|   Matei Zaharia|               1|[500, 250, 100]|100|   Contributor|
|       2|Michael Armbrust|               1|     [250, 100]|250|    PMC Member|
|       2|Michael Armbrust|               1|     [250, 100]|100|   Contributor|
+--------+----------------+----------------+---------------+---+--------------+



**User Defined Functions**

We also define and use our own custom functions within transformations (in expressions). 


In [35]:
from pyspark.sql.functions import col
from pyspark.sql.functions import udf
# define the function
def power3(double_value):
    return double_value ** 3
# register the function
power3udf = udf(power3)
# use the function with SQL queries
udfExampleDF = spark.range(10).toDF("num")
udfExampleDF.select(col("num"), power3udf(col("num"))).show()

+---+-----------+
|num|power3(num)|
+---+-----------+
|  0|          0|
|  1|          1|
|  2|          8|
|  3|         27|
|  4|         64|
|  5|        125|
|  6|        216|
|  7|        343|
|  8|        512|
|  9|        729|
+---+-----------+



In [36]:
# Stop the Spark context
spark.stop()